### Coleta do Ranking de Municípios por Número de Salas (2004 - 2020)

Nesta etapa, importamos as bibliotecas necessárias e configuramos o Selenium para automatizar a extração.

O código abaixo utiliza um laço de repetição (`for`) para buscar os dados ano a ano (de 2020 até 2004, em ordem decrescente). Como a estrutura do site e os textos das tabelas mudaram ao longo do tempo, implementamos algumas regras (`if / elif`):
1. **Texto de Busca:** Muda dependendo do intervalo de anos para encontrar a tabela correta no HTML.
2. **Recorte da Tabela (`iloc` e `drop`):** Apara as colunas excedentes e as linhas iniciais de sujeira de forma dinâmica, pois o layout da tabela de 2007 em diante é diferente dos anos anteriores.

Por fim, os dados extraídos de cada ano são salvos em arquivos `.csv` separados. Utilizamos um bloco `try...except` para garantir que, se a coleta falhar em um ano específico, o loop não seja interrompido e continue para os anos seguintes.

In [ ]:
%load_ext autoreload
%autoreload 2

import time
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd

from constantes import (
    COLETA_RANKING_MUNICIPIOS_2020_2007,
    COLETA_RANKING_MUNICIPIOS_2006_2005,
    COLETA_RANKING_MUNICIPIOS_2004
)

from funcoes import coleta_ranking_municipios

# Configura o navegador
options = webdriver.ChromeOptions()
meu_navegador = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

for ano in range(2020, 2003, -1):

    if ano >= 2007:
        texto_busca = COLETA_RANKING_MUNICIPIOS_2020_2007
    elif ano >= 2005:
        texto_busca = COLETA_RANKING_MUNICIPIOS_2006_2005
    elif ano == 2004:
        texto_busca = COLETA_RANKING_MUNICIPIOS_2004


    df_temp = coleta_ranking_municipios(meu_navegador, texto_busca, ano)

    try:
        if df_temp is not None:

            if ano >= 2007:
                df_temp = df_temp.iloc[7:, 1:5]
                df_temp = df_temp.drop(3, axis=1)
            else:
                df_temp = df_temp.iloc[7:, 1:4]

            df_temp.to_csv(f'ranking_municipios_{ano}.csv', index=False)
            
    except Exception as erro_real:
        print(f"Ano {ano} falhou. O verdadeiro erro é: {erro_real}")
        

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Após a execução do script de extração, fazemos um teste rápido para verificar se a coleta foi bem-sucedida. 
Aqui, utilizamos a função `pd.read_csv()` para carregar os dados salvos do ano de 2020 e exibi-los na tela. Note que os dados brutos ainda trazem algumas linhas indesejadas no início e no final (como "Fonte" e "Observações"), que deverão ser tratadas na fase de limpeza.

In [ ]:
df_teste = pd.read_csv("dados/ranking_municipios_2020.csv")
df_teste

,1,2,4
0,município,UF,salas
1,SÃO PAULO,SP,350
2,RIO DE JANEIRO,RJ,219
3,CURITIBA,PR,92
4,BRASÍLIA *,DF,88
...,...,...,...
449,VOTUPORANGA,SP,1
450,XANGRI-LÁ,RS,1
451,Fonte: Filme B Box Office Brasil - Pesquisa: F...,Fonte: Filme B Box Office Brasil - Pesquisa: F...,Fonte: Filme B Box Office Brasil - Pesquisa: F...
452,"Observações: * Segundo o IBGE, Brasília é o ún...","Observações: * Segundo o IBGE, Brasília é o ún...","Observações: * Segundo o IBGE, Brasília é o ún..."
